# Baseline｜等权、滚动 ICIR 与 LightGBM 样本外评估

本 Notebook 只读取已经冻结且通过完整性校验的 OOS evaluation artifact，不重新训练模型、不重新生成 Test scores，也不改变任何策略或统计定义。报告比较等权、滚动 ICIR 和 LightGBM 三种策略在 2021–2025 年样本外区间的表现；滚动 ICIR 每20个交易日仅使用已经完整实现的历史标签更新。

阅读顺序为：样本覆盖 → 评分有效性 → 分组单调性 → 多头与超额收益 → 换手率 → LightGBM 跨阶段稳定性 → 冻结信息。

## 00｜参数与权威性校验

这里绑定本次正式 Baseline 的 OOS evaluation manifest，以及完整 Factor Pool、Static Strategy Bundle 和 Test Score Artifact 的权威指纹。任一指纹不一致都会拒绝加载。输出目录按 evaluation fingerprint 隔离，避免覆盖其他实验。

In [ ]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from factor_gfn.backtest import load_verified_oos_baseline_evaluation
from factor_gfn.reporting import build_oos_report_data, OOSReportRenderer

LATEST_POINTER = Path(r'runs/oos_baseline_evaluations/LATEST_ROLLING_ICIR_MANIFEST.txt')
if not LATEST_POINTER.is_file():
    raise FileNotFoundError('请先运行 run_baseline_freeze_and_oos.ipynb 的滚动 ICIR OOS Cell')
EVALUATION_MANIFEST = Path(LATEST_POINTER.read_text(encoding='utf-8').strip())
manifest_identity = json.loads(EVALUATION_MANIFEST.read_text(encoding='utf-8'))
FACTOR_POOL_FINGERPRINT = manifest_identity['factor_pool_fingerprint']
STRATEGY_BUNDLE_FINGERPRINT = manifest_identity['strategy_bundle_fingerprint']
TEST_SCORE_ARTIFACT_FINGERPRINT = manifest_identity['test_score_artifact_fingerprint']
OUTPUT_ROOT = Path(r'outputs/oos_baseline')

evaluation = load_verified_oos_baseline_evaluation(
    EVALUATION_MANIFEST,
    expected_factor_pool_fingerprint=FACTOR_POOL_FINGERPRINT,
    expected_strategy_bundle_fingerprint=STRATEGY_BUNDLE_FINGERPRINT,
    expected_test_score_artifact_fingerprint=TEST_SCORE_ARTIFACT_FINGERPRINT,
)
report = build_oos_report_data(evaluation)
renderer = OOSReportRenderer(report, OUTPUT_ROOT / evaluation.fingerprint)

中文列名 = {
    'Strategy': '策略', 'Sample': '样本', 'Split': '阶段',
    'Mean RankIC': '平均 RankIC', 'ICIR': 'ICIR',
    'G10 Geometric Annualized Return': 'G10 几何年化收益率',
    'G10 Annualized Return': 'G10 年化收益率',
    'G10 Annualized Volatility': 'G10 年化波动率',
    'G10 Sharpe': 'G10 夏普比率', 'G10 Max Drawdown': 'G10 最大回撤',
    'Excess Geometric Annualized Return': '超额几何年化收益率',
    'Excess Annualized Return': '超额年化收益率',
    'Excess Annualized Volatility': '超额年化波动率',
    'Excess Sharpe': '超额夏普比率', 'Excess Max Drawdown': '超额最大回撤',
    'Excess Win Rate': '超额胜率',
    'Mean One-Way Turnover': '平均单边换手率',
    'Median': '中位数', 'Max': '最大值', 'Turnover Observations': '换手观测数',
    'Mean Constituent Replacement Rate': '平均成分替换率',
    'Mean Raw Universe Count': '平均原始股票数',
    'Mean Complete-Case Eligible Count': '平均基础合格股票数',
    'Mean Label-Eligible Count': '平均标签有效股票数',
    'Min Eligible Count': '最少基础合格股票数',
    'Mean Coverage Ratio': '平均覆盖率', 'Min Coverage Ratio': '最低覆盖率',
    'Invalid Rebalance Periods': '无效调仓期数',
    'Field': '字段', 'Value': '取值',
}
中文取值 = {
    'Equal Weight': '等权', 'Rolling ICIR': '滚动 ICIR',
    'Train': '训练集', 'Validation': '验证集', 'Test': '测试集',
    'common_sample': '统一样本',
}

def 显示中文表格(dataframe):
    return dataframe.rename(columns=中文列名).replace(中文取值)

print('OOS 评估状态：', evaluation.manifest['evaluation_status'])
print('OOS 评估指纹：', evaluation.fingerprint)
print('样本外日期：', evaluation.manifest['key_ranges']['coverage_by_date'])
print('报告输出目录：', renderer.output_dir)


> **当前策略矩阵口径**：Raw expression → 1%/99% 截面缩尾 → PIT 申万一级行业中性化 → 截面标准化 → base-eligible 股票内的因子特定缺失值填 0 → 冻结 Train direction → Top100 Strategy Matrix。
>
> 当前 evaluation manifest 和底层 reporting 数据中仍保留旧字段名 `complete_case` / `Mean Complete-Case Eligible Count`，这是兼容性命名残留；本 Notebook 将其显示为“基础合格股票”，实际结果并不是重新要求 Top100 全部同时非缺失。

## 01｜样本覆盖

该图回答策略在每个调仓日实际覆盖了多少只股票。上图比较原始股票池与满足基础资格条件的股票数量，下图展示覆盖率。重点检查是否存在覆盖率突然接近 0、股票数异常断层或大量无效调仓期。这里的基础资格主要由正式股票池和可用的 PIT 行业标签决定，因子特定缺失已经在 cleaning 后填 0。

In [ ]:
renderer.figure_coverage()

In [ ]:
显示中文表格(report.coverage_summary)

## 02｜策略评分有效性

周期 RankIC 衡量每个调仓截面中策略分数与未来收益排序的一致性；累计 IC 只是 RankIC 的累计和，不是净值。策略分数相关性热力图用于判断三种策略是否实际上给出了高度相似的股票排序。相关性高代表信号冗余较强，但不能单独判断哪种策略更有效。热力图保留三位小数，避免把接近 1 的相关性误读为严格等于 1；下方同时显示四位小数的精确矩阵。

In [ ]:
for 策略编号 in ('equal_weight', 'fixed_icir', 'lightgbm'):
    display(renderer.figure_rank_ic(策略编号))

In [ ]:
renderer.figure_score_correlation()

In [ ]:
相关矩阵中文 = report.strategy_score_correlation.rename(
    index={'Equal Weight': '等权', 'Rolling ICIR': '滚动 ICIR'},
    columns={'Equal Weight': '等权', 'Rolling ICIR': '滚动 ICIR'},
)
display(相关矩阵中文.style.format('{:.4f}'))

## 03｜十分位组合分析

每个调仓日按策略分数从低到高分为 G1–G10，G10 代表策略最看好的股票。主图展示各组未来 5 日收益减去同日 evaluation-eligible 股票池等权基准后的平均超额收益；三个策略使用完全相同的基准，0 轴用于判断各分组相对同日市场截面的超额方向。判断横截面区分能力时，应联合观察分组单调性与 G10−G1。原始绝对收益仍保留在冻结 OOS artifact 中，本图仅改变展示口径。

In [ ]:
renderer.figure_decile_return()

In [ ]:
显示中文表格(report.decile_return_table)

In [ ]:
策略中文名 = {'equal_weight': '等权', 'fixed_icir': '滚动 ICIR', 'lightgbm': 'LightGBM'}
分组列 = [f'G{i}' for i in range(1, 11)]
平均分组绝对收益 = report.decile_returns.groupby('strategy_id')[分组列].mean()
平均同期基准收益 = report.portfolio_returns.groupby('strategy_id')['benchmark_return'].mean()
平均分组超额收益 = 平均分组绝对收益.sub(平均同期基准收益, axis=0).rename(index=策略中文名)
print('各组相对同日全样本等权基准的平均 5 日超额收益：')
display(平均分组超额收益.style.format('{:.2%}').background_gradient(cmap='RdYlGn', axis=None))

## 04｜G10 多头组合与基准

这里比较三种策略各自 G10 多头组合的累计净值，并加入同一合格股票样本上的等权基准。该图展示的是未扣交易成本的毛收益净值，因此需要结合后续换手率判断策略在实际交易中的可实现性。

In [ ]:
renderer.figure_g10_nav()

## 05｜超额收益与多空收益

超额净值定义为 G10 相对统一等权基准的累计表现；多空净值定义为 G10−G1。前者更接近多头选股价值，后者更集中反映评分横截面区分能力。主绩效表汇总 RankIC、ICIR、年化收益、波动率、最大回撤、胜率和换手率。

In [ ]:
renderer.figure_excess_nav()

In [ ]:
renderer.figure_long_short_nav()

In [ ]:
显示中文表格(report.main_strategy_performance_summary)

### 三种冻结策略样本外绩效对比

该图把三种策略在完全相同 Test 样本上的核心指标放入同一色阶表，并由正式 renderer 写入 figures。每一列独立着色；使用浅红—柔黄—浅绿的压缩渐变，避免把相对较弱但仍可接受的结果渲染成强烈红色。除换手率越低越好外，其余列均按数值越高越好处理。最大回撤为负数，因此更接近 0 的值颜色更优。

In [ ]:
renderer.figure_performance_comparison()

### 分年度收益与超额净值回撤

三种策略分别作图，不把不同策略混在同一坐标轴。年度图按照参考样式，每年仅展示一根复利超额收益柱，正值为蓝色、负值为红色；超额净值图上方使用蓝色多头超额净值曲线，下方使用淡红色回撤曲线与区域。

In [ ]:
for 策略编号 in ('equal_weight', 'fixed_icir', 'lightgbm'):
    display(renderer.figure_annual_returns(策略编号))

In [ ]:
for 策略编号 in ('equal_weight', 'fixed_icir', 'lightgbm'):
    display(renderer.figure_excess_nav_drawdown(策略编号))

## 06｜换手率

换手率采用持仓漂移调整后的单边口径。平均换手率用于比较三种策略的总体交易强度，时间序列用于识别阶段性跳升、组合重置或不稳定时期。首期以及重置期可能为 NaN，这是口径设计而不是计算错误。

In [ ]:
renderer.figure_average_turnover()

In [ ]:
renderer.figure_turnover_series()

In [ ]:
显示中文表格(report.turnover_summary)

## 07｜滚动 ICIR 权重诊断

主方案使用最近150个五日频IC观察，每4个调仓周期更新一次；任何IC只有在对应 `open[t+6]` 已经可得后才允许进入历史窗口。下图用于检查单因子权重上限、集中度以及参与动态加权的因子数量。

In [ ]:
renderer.figure_rolling_icir()

In [ ]:
显示中文表格(report.rolling_icir_diagnostics_by_update)

## 08｜LightGBM 跨阶段诊断

该图并列展示 LightGBM 在训练集、验证集和真实测试集上的平均 RankIC 与 ICIR。验证集承担 Development 阶段早停的角色，因此不是最终 OOS；测试集才是模型冻结后首次访问的真实 OOS。重点观察方向是否一致以及从 Development 到 Test 的衰减程度。

In [ ]:
renderer.figure_lightgbm_splits()

In [ ]:
显示中文表格(report.lightgbm_split_effectiveness)

## 09｜策略冻结与可复现信息

本表记录 Factor Pool、Strategy Bundle、Test Score Artifact、滚动 ICIR 配置与权重路径及 LightGBM 模型的关键身份信息。滚动 ICIR 的规则和开发期种子在读取 Test labels 前冻结，后续仅按成熟标签因果更新。

In [ ]:
显示中文表格(report.strategy_freeze_summary)

## 10｜导出正式报告

最后统一生成全部图、CSV 表格和 report manifest。前面的单图 Cell 仅用于交互查看；本 Cell 才是完整报告导出入口。导出不会重新训练策略或重新计算 Test scores。

In [ ]:
report_manifest_path = renderer.render_all()
print('正式 OOS 报告 manifest：', report_manifest_path)
print('图表目录：', report_manifest_path.parent / 'figures')
print('表格目录：', report_manifest_path.parent / 'tables')
report_manifest_path